In [ ]:
import gzip
import struct
import numpy as np
from scipy.spatial.distance import cdist
from scipy.linalg import eigh
import matplotlib.pyplot as plt
import pandas as pd

# ── STEP 1: PARSE AND DECOMPRESS YOUR .RDATA DATASET ────────────────────────
def load_sardine_data(filepath):
    """Custom parser to decompress and extract the California vectors from the binary RData file."""
    with gzip.open(filepath, 'rb') as f:
        raw = f.read()
    N = 78
    years   = np.array([struct.unpack_from('>i', raw,  70 + i*4)[0] for i in range(N)])
    anchovy = np.array([struct.unpack_from('>d', raw, 390 + i*8)[0] for i in range(N)])
    sardine = np.array([struct.unpack_from('>d', raw, 1022 + i*8)[0] for i in range(N)])
    sio_sst = np.array([struct.unpack_from('>d', raw, 1654 + i*8)[0] for i in range(N)])
    np_sst  = np.array([struct.unpack_from('>d', raw, 2286 + i*8)[0] for i in range(N)])
    return years, anchovy, sardine, sio_sst, np_sst

my_path = r'C:\Users\noegi\Downloads\sardine_anchovy_sst (1).RData'
years, anchovy, sardine, sio_sst, np_sst = load_sardine_data(my_path)
print(f"✅ Successfully loaded {len(years)} annual observations ({int(years[0])}–{int(years[-1])})")


# ── STEP 2: Z-SCORE STANDARDIZATION ──────────────────────────────────────────
def zscore(ts):
    return (ts - np.mean(ts)) / np.std(ts)

anchovy_n = zscore(anchovy)
sardine_n = zscore(sardine)
sio_sst_n = zscore(sio_sst)
np_sst_n  = zscore(np_sst)


# ── STEP 3: LOCALIZED MANIFOLD ENGINE ────────────────────────────────────────
def create_recurrence_matrix(ts, E=3, tau=1, k=10):
    """Creates a localized k-NN affinity matrix optimized for small historical vectors."""
    n = len(ts)
    n_embed = n - (E - 1) * tau
    M = np.array([[ts[i + (E-1)*tau - j*tau] for j in range(E)] for i in range(n_embed)])
    dists = cdist(M, M, metric='euclidean')

    A = np.zeros_like(dists)
    for i in range(n_embed):
        nn_idx = np.argsort(dists[i])[1:k+1]
        sigma = np.mean(dists[i, nn_idx]) + 1e-12
        A[i, nn_idx] = np.exp(-dists[i, nn_idx]**2 / (2 * sigma**2))

    return (A + A.T) / 2


def shrec_reconstruct(ts_list, E=3, tau=1, k=10, n_components=2):
    """SHREC pipeline for a list of time series."""
    A_list = [create_recurrence_matrix(ts, E=E, tau=tau, k=k) for ts in ts_list]
    A_consensus = np.sum(A_list, axis=0)

    degrees = np.sum(A_consensus, axis=1)
    degrees[degrees == 0] = 1e-12
    D_inv_sqrt = np.diag(1.0 / np.sqrt(degrees))

    L = np.eye(len(A_consensus)) - D_inv_sqrt @ A_consensus @ D_inv_sqrt
    eigenvalues, eigenvectors = eigh(L)

    idx = np.argsort(eigenvalues)
    eigenvectors = eigenvectors[:, idx]

    return eigenvectors[:, 1:1+n_components]


# ── STEP 4: DEFINE ALL 6 PAIRWISE COMBINATIONS ───────────────────────────────
E_param = 3
time_axis = years[(E_param - 1):]

pairs = [
    (anchovy_n, np_sst_n,  "Anchovy & NP SST"),
    (anchovy_n, sio_sst_n, "Anchovy & SIO SST"),
    (anchovy_n, sardine_n, "Anchovy & Sardine"),
    (sardine_n, np_sst_n,  "Sardine & NP SST"),
    (sardine_n, sio_sst_n, "Sardine & SIO SST"),
    (np_sst_n,  sio_sst_n, "NP SST & SIO SST"),
]


# ── STEP 5: RUN SHREC AND PLOT EACH PAIR ─────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

window_size = 5
half_w = window_size // 2

all_psi1 = {}

for ax, (ts_a, ts_b, label) in zip(axes, pairs):
    latent_modes = shrec_reconstruct([ts_a, ts_b])
    psi_1 = zscore(latent_modes[:, 0])

    all_psi1[label] = psi_1

    ax.plot(time_axis, psi_1, color='teal', alpha=0.4, label='Raw $\\psi_1$')

    smoothed = np.convolve(psi_1, np.ones(window_size) / window_size, mode='same')
    ax.plot(time_axis[half_w:-half_w], smoothed[half_w:-half_w],
            color='darkcyan', linewidth=2.5, label='5-Yr Trend')

    ax.set_title(label)
    ax.set_xlabel('Year')
    ax.set_ylabel('$\\psi_1$ (standardized)')
    ax.grid(True, linestyle='--')
    ax.legend(fontsize=8)

plt.suptitle('SHREC Pairwise Latent Trajectories ($\\psi_1$)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


# ── STEP 6: SAVE ALL PAIRWISE PSI_1 TO CSV ───────────────────────────────────
print("\nStep 6: Saving pairwise SHREC latent data ...")

export_data = {'Year': time_axis.astype(int)}
for label, psi_1 in all_psi1.items():
    col_name = label.replace(' ', '_').replace('&', 'and')
    export_data[f'Psi1_{col_name}'] = psi_1

shrec_df = pd.DataFrame(export_data)
output_csv = "shrec_pairwise_latent_coordinates.csv"
shrec_df.to_csv(output_csv, index=False)

print(f"💾 Successfully saved {len(shrec_df)} rows to '{output_csv}'!")
print("   Columns exported:", list(shrec_df.columns))